# Vector Storage with FAISS, Chroma, and Pinecone

## What we will learn:
- How to efficiently store and search embeddings with FAISS and Chroma (local) and Pinecone (cloud).
- The importance of metadata for precise filtering.
- Practical differences between Flat and HNSW indices.
- How to migrate from a local solution to a managed cloud solution.

### Why is vector storage crucial?
- **Scale**: Search through millions of documents in milliseconds.
- **Persistence**: Avoid recalculating embeddings on every execution.
- **Filters**: Combine semantic search with business rules.
- **Production**: Cloud solutions like Pinecone offer scalability and simplified management.

## Environment Setup

**Important:** For the Pinecone section, you will need an account and an API key.
1. Create an account at [pinecone.io](https://www.pinecone.io/).
2. Create an index (e.g., `langchain-rag`) with the correct dimension for the embeddings.
3. In the "API Keys" section, create a key.

Add your keys to the `.env` file:
```
PINECONE_API_KEY="your-pinecone-key-here"
```

In [1]:
!pip install langchain openai faiss-cpu chromadb langchain-pinecone

!pip install langchain-community

## Setup - Load API Key and Initialize Client

In [2]:
from dotenv import load_dotenv
import os

# Load the .env file
load_dotenv(dotenv_path='../../.env')  # Specify the path to your .env file

# Access the environment variable
api_key = os.getenv('OPENAI_API_KEY')

# Check if the variable is loaded
if api_key or api_key == "":
    print("API key loaded successfully.")
else:
    print("Failed to load API key.")

from openai import OpenAI
client = OpenAI(api_key=api_key)

API key loaded successfully.


In [3]:
from langchain_openai import OpenAIEmbeddings
from langchain.schema import Document

embeddings = OpenAIEmbeddings(openai_api_key=api_key)

/Users/julio.cesar/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


------
## NOTE:

Let's break down the last line in more detail:

```python
embeddings = OpenAIEmbeddings(openai_api_key=api_key)
```

### ✅ What does this line do?

It **instantiates** an **embedding model** from **OpenAI** (via LangChain), allowing the transformation of texts into **high-dimensional numerical vectors**.

---

### 🔍 Key Concepts:

#### 🔹 `OpenAIEmbeddings`

* It is a **class from LangChain** that connects to the **OpenAI embedding service**.
* It transforms **texts** into **numerical vectors**, which can be used for **semantic searches**, **text similarity**, or **vector indexing** (in FAISS, Pinecone, ChromaDB, etc.).

---

-----


### 🧠 Practical example:


In [4]:
embeddings.embed_query("Company's home office policy")

[-0.012550236657261848,
 -0.010249132290482521,
 -0.004202758893370628,
 -0.021877560764551163,
 -0.014639666303992271,
 0.00861036404967308,
 -0.021659059450030327,
 0.006073686294257641,
 0.014694292098283768,
 -0.0302899070084095,
 0.02038901299238205,
 -0.01145089603960514,
 0.00578690180554986,
 -0.004066194873303175,
 -0.014393851161003113,
 0.03340356796979904,
 0.015540989115834236,
 -0.03941238671541214,
 0.024294745177030563,
 -0.004660248290747404,
 -0.02881501615047455,
 0.008624020032584667,
 -0.008958602324128151,
 0.010419837199151516,
 -0.006947696208953857,
 -0.01685200445353985,
 0.010733935050666332,
 -0.020416326820850372,
 0.004704631865024567,
 0.0036906434688717127,
 0.03384057432413101,
 -0.014407508075237274,
 -0.0006205129320733249,
 0.007777323015034199,
 -0.015568302012979984,
 0.004619278945028782,
 -0.011246049776673317,
 -0.0013895392185077071,
 0.026643646880984306,
 -0.030180655419826508,
 0.027244528755545616,
 0.013444731011986732,
 -0.002806391334161

## Preparing Documents with Metadata

Metadata is the key to filtered and contextual searches. Let's create documents rich in metadata.

Let's **create a list of structured documents**, each with:

* A **textual content** (called `page_content`)
* A **set of metadata** (such as type, department, year, etc.)

These documents are of type `Document`, a **LangChain class** used to store **texts and their metadata** in a standardized way — ideal for RAG, semantic search, vector indexing, etc.

---

### 🧠 **What is this for?**

This type of structure is **fundamental** in RAG (Retrieval-Augmented Generation) pipelines, where:

* The **text** will be transformed into **vectors** (with embeddings)
* The **metadata** helps to:

  * Filter by area (e.g., HR, IT)
  * Know the content's date
  * Trace which document generated the response

---

### 🧩 **Practical example:**

You could use this list in an intelligent search mechanism, like this:

* 🔎 "What is the deadline to request vacation?"

  * The system finds the `Document` whose content talks about vacations
  * And returns the answer based on that text (e.g., via RAG)

---

### 🔍 Structure of a `Document`:

```python
Document(
    page_content="Text that describes the informative content",
    metadata={
        "key1": "value",
        "key2": "value",
        ...
    }
)
```

---

### ✅ Utility in real projects:

* 🔍 Question and answer systems (e.g., internal chatbot)
* 📁 Search in corporate documents
* 🤖 Applications with RAG and generative AI
* 🗂️ Organization and retrieval of knowledge



In [5]:
company_documents = [
    Document(
        page_content="Vacation policy: Employees are entitled to 30 days of vacation after 12 months. The request must be made 30 days in advance.",
        metadata={"type": "policy", "department": "HR", "year": 2024, "id_doc": "doc001"}
    ),

    Document(
        page_content="Expense reimbursement process: Submit the invoice through the financial portal. Reimbursement occurs within 5 business days.",
        metadata={"type": "process", "department": "Finance", "year": 2023, "id_doc": "doc002"}
    ),

    Document(
        page_content="IT Guide: To set up the VPN, visit vpn.ourcompany.com and follow the instructions for your operating system.",
        metadata={"type": "tutorial", "department": "IT", "year": 2024, "id_doc": "doc003"}
    ),

    Document(
        page_content="Code of Ethics and Conduct: We value respect, integrity, and collaboration. Cases of harassment will not be tolerated.",
        metadata={"type": "policy", "department": "HR", "year": 2022, "id_doc": "doc004"}
    )

]

OBS:In a real environment, you wouldn't manually write the documents as in the previous example, they would come, for example, from a PDF, a database, a folder with files, spreadsheets, internal sites, etc.


-------

## FAISS - Local Performance

**What is FAISS?**
It is an open-source library for very fast similarity searches. Given an item, it finds the most similar ones within a large volume of data.

FAISS mainly offers two search strategies:

#### 1. Flat Index (Exact Search)

* **How it works:** Compares your search item with **all** other items in the database, one by one.
* **Advantage:** 100% accuracy. You are guaranteed to find the exact results.
* **Disadvantage:** Slow for many data, as the number of comparisons is huge.

#### 2. HNSW Index (Approximate Search)

* **How it works:** Creates an optimized data structure (a graph) that allows skipping unnecessary comparisons. The search is intelligently guided to the most likely area of the results.
* **Advantage:** Extremely faster, ideal for real-time applications with large data volumes.
* **Disadvantage:** The accuracy is not 100%. The search is very good, but may, rarely, leave out the most exact result. This loss is generally acceptable in exchange for speed.

**Conclusion about LangChain:**
By default, LangChain uses the **Flat** method. It chooses safety (100% accuracy) over speed, as it is simpler and more reliable for initial or small projects.

LangChain uses `IndexFlatL2` by default in FAISS.


### `IndexFlatL2`

* **What it is:** **Exact** Search (Brute Force).
* **How it works:** Compares your search item with **all** the others.
* **Pro:** Guarantees **100% accuracy**.
* **Con:** **Slow** for large data volumes.
* **Use for:** Small datasets or to ensure the perfect result without worrying about speed.

### `IndexHNSWFlat`

* **What it is:** **Approximate** Search (Intelligent and Fast).
* **How it works:** Uses a graph (a "map") to efficiently navigate and find the most likely results without looking at everything.
* **Pro:** **Extremely fast**, even with millions of items.
* **Con:** Accuracy is not 100% (but usually above 99%) and consumes more memory.
* **Use for:** Most production applications, where **speed** is more important than perfect accuracy.

In [6]:
from langchain_community.vectorstores import FAISS
import faiss

d = 768
index_hnsw = faiss.IndexHNSWFlat(d, 32)

In [7]:
faiss_db = FAISS.from_documents(company_documents, embeddings)

question = "How do I request my vacation?"
results = faiss_db.similarity_search(question, k=2)

In [8]:
print(f"\n🔍 Question: '{question}'")
print("\n Relevant documents (FAISS):")
for doc in results:
    print(f"- {doc.page_content}")
    print(f" (Metadata: {doc.metadata})")


🔍 Question: 'How do I request my vacation?'

 Relevant documents (FAISS):
- Vacation policy: Employees are entitled to 30 days of vacation after 12 months. The request must be made 30 days in advance.
 (Metadata: {'type': 'policy', 'department': 'HR', 'year': 2024, 'id_doc': 'doc001'})
- Expense reimbursement process: Submit the invoice through the financial portal. Reimbursement occurs within 5 business days.
 (Metadata: {'type': 'process', 'department': 'Finance', 'year': 2023, 'id_doc': 'doc002'})


## ChromaDB - Powerful Filters

[Chroma](https://www.trychroma.com/)

**Chroma DB: The Simple and Powerful Vector Database for AI**

Chroma is an open-source vector database, specially created to be intuitive and easy to use in Artificial Intelligence applications.

Its great differential is the combination of two essential functions:

* **Similarity Search:** Stores and searches vectors (the numerical representation of texts, images, etc.) to find items similar based on their meaning or content.
* **Metadata Filters:** Allows associating additional information (such as dates, categories, sources, user IDs) to each vector. With this, you can refine your searches precisely.

**In practice,** this allows complex queries like: "Find documents *similar to this text about finance*, but that were published *only in the last month* and belong to the *'news' category*."

---

In [9]:
from langchain_community.vectorstores import Chroma

chroma_db = Chroma.from_documents(
    documents=company_documents,
    embedding=embeddings
)

In [10]:
results = chroma_db.similarity_search(question, k=2)

for doc in results:
  print(f"- {doc.page_content}")

- Vacation policy: Employees are entitled to 30 days of vacation after 12 months. The request must be made 30 days in advance.
- Expense reimbursement process: Submit the invoice through the financial portal. Reimbursement occurs within 5 business days.


In [11]:
hr_question = "What are the company's rules?"

filtered_results = chroma_db.similarity_search(
    hr_question,
    k=2,
    filter={"$and": [{"department": "HR"}, {"type": "policy"}]}
)

In [12]:
print(f"\n Question: '{hr_question}' with filter for HR policies")
print("\n Relevant and filtered documents (Chroma):")

for doc in filtered_results:
  print(f"- {doc.page_content}")
  print(f"  (Department: {doc.metadata['department']}, Type: {doc.metadata['type']})")


 Question: 'What are the company's rules?' with filter for HR policies

 Relevant and filtered documents (Chroma):
- Code of Ethics and Conduct: We value respect, integrity, and collaboration. Cases of harassment will not be tolerated.
  (Department: HR, Type: policy)
- Vacation policy: Employees are entitled to 30 days of vacation after 12 months. The request must be made 30 days in advance.
  (Department: HR, Type: policy)
